# Deep Q-Networks (DQN)

_Modern Deep Learning & AI_

**Let a neural net guess the value of each move. Train it to match reward-now plus the best value next.**

Q-learning keeps a value $Q(s,a)$ for every state-action pair. But real games have far too many states to list.

     A Deep Q-Network replaces that giant table with a neural network. The net reads a state and outputs a $Q$ value for each action.

---

This notebook is a practice scaffold for the **Deep Q-Networks (DQN)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

class QNet(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32), nn.ReLU(),
            nn.Linear(32, n_actions))            # one Q value per action
    def forward(self, s):
        return self.net(s)

state_dim, n_actions, gamma = 4, 2, 0.9
q_net, target_net = QNet(state_dim, n_actions), QNet(state_dim, n_actions)
target_net.load_state_dict(q_net.state_dict())   # target = slow copy
opt = torch.optim.Adam(q_net.parameters(), lr=1e-3)

# a synthetic mini-batch of transitions (s, a, r, s', done)
s  = torch.randn(8, state_dim)
a  = torch.randint(0, n_actions, (8, 1))
r  = torch.randn(8)
s2 = torch.randn(8, state_dim)
done = torch.zeros(8)

q_sa = q_net(s).gather(1, a).squeeze(1)          # Q(s, a) taken
with torch.no_grad():
    max_q2 = target_net(s2).max(dim=1).values    # max_a' Q_target(s')
    y = r + gamma * (1 - done) * max_q2          # TD target
loss = nn.functional.mse_loss(q_sa, y)           # (y - Q(s,a))^2
opt.zero_grad(); loss.backward(); opt.step()
print(float(loss))

## Visualize the data & results

_On an 8-state corridor task, is the Q-learning agent actually learning? Does episode reward rise?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
N, GOAL = 8, 7                       # corridor of 8 states, goal at the right end
Q = np.zeros((N, 2))                 # tabular Q: 2 actions (left=0, right=1)
gamma, alpha = 0.95, 0.2
rewards = []
for ep in range(300):
    s, total, eps = 0, 0.0, max(0.05, 1.0 - ep / 150)   # epsilon decay
    for _ in range(30):
        greedy = rng.random() >= eps                     # explore vs exploit
        a = int(np.argmax(Q[s])) if greedy else rng.integers(2)
        s2 = min(N - 1, s + 1) if a == 1 else max(0, s - 1)
        r = 10.0 if s2 == GOAL else -0.1                 # +10 at goal
        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
        s, total = s2, total + r
        if s2 == GOAL:
            break
    rewards.append(total)

sm = np.convolve(rewards, np.ones(20) / 20, mode='valid')   # smoothed curve
plt.figure(figsize=(6, 4))
plt.plot(sm, color='#4ea1ff', label='episode reward')
plt.xlabel('episode'); plt.ylabel('smoothed episode reward')
plt.title('Tabular Q-learning on 8-state corridor'); plt.legend()
plt.tight_layout(); plt.show()